In [2]:
import numpy as np

In [3]:
#load data
X = np.load("../data/savedData/X_data.npy", allow_pickle=True)
y = np.load("../data/savedData/y_data.npy", allow_pickle=True)
regions = np.load("../data/savedData/regions.npy", allow_pickle=True)

In [5]:
print(X.shape)
print(np.unique(y, return_counts=True))

(73492, 60, 38)
(array([0, 1]), array([72238,  1254]))


In [3]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_le = le.fit_transform(y)
# print(le.classes_)
# print(np.unique(y_le, return_counts=True))


['B' 'C' 'M' 'NF' 'X']
(array([0, 1, 2, 3, 4]), array([ 5692,  6416,  1089, 60130,   165]))


In [4]:
# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y_le, test_size=0.4, random_state=42, stratify=y_le)

In [ ]:
# np.save("X_train.npy", X_train)
# np.save("X_test.npy", X_test)
# np.save("y_train.npy", y_train)
# np.save("y_test.npy", y_test)

In [5]:
# X_test = np.load("../notebooks/X_test.npy", allow_pickle=True)
# X_train = np.load("../notebooks/X_train.npy", allow_pickle=True)
# y_test = np.load("../notebooks/y_test.npy", allow_pickle=True)
# y_train = np.load("../notebooks/y_train.npy", allow_pickle=True)

In [7]:
#group by active region
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5)

for train_index, test_index in sgkf.split(X, y, regions):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]
    break

In [10]:
#allow balanced weighting between flare classes
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
classWeighting = compute_class_weight(
    class_weight= "balanced",
    classes=classes,
    y=y_train
)

In [11]:
classWeight = dict(zip(classes, classWeighting))

In [8]:
# conda install -c conda-forge sktime

In [ ]:
# pip install sktime

In [6]:
# # # Balanced dataset

# from collections import Counter
# from imblearn.under_sampling import RandomUnderSampler

# samples, rows, feat = X_train.shape

# X_train_2d = X_train.reshape(samples, -1)


# rus = RandomUnderSampler(random_state=42, sampling_strategy={0:10000})
# X_balanced_2d, y_balanced = rus.fit_resample(X_train_2d, y_train)

# X_balanced = X_balanced_2d.reshape(-1, rows, feat)



In [13]:
from sktime.transformations.panel.rocket import MiniRocket

In [14]:
#minirocket parameters
miniRock = MiniRocket(
    num_kernels=300,
    random_state=42,
    n_jobs = -1
)


In [8]:
# # --- Extract minority class ---
# X_min = X_train[y_train == 1]
# y_min = y_train[y_train == 1]

# # --- Decide how many copies you want ---
# copies = 5   # try 3, 5, 8 etc.

# noise_level = 0.01

# X_aug_list = [X_train]
# y_aug_list = [y_train]

# for _ in range(copies):
#     noise = noise_level * np.random.randn(*X_min.shape)
#     X_min_noisy = X_min + noise
    
#     X_aug_list.append(X_min_noisy)
#     y_aug_list.append(y_min)

# # --- Concatenate everything ---
# X_train_aug = np.concatenate(X_aug_list, axis=0)
# y_train_aug = np.concatenate(y_aug_list, axis=0)

In [ ]:
#gaussian noise implementation

In [15]:
X_min = X_train[y_train == 1]

In [16]:
noise_level = 0.018

In [17]:
noise = noise_level * np.random.randn(*X_min.shape)
np.random.seed(42)
X_min_noise = X_min + noise

In [18]:
y_min_noise = np.ones(len(X_min_noise))

In [19]:
X_train_augm = np.concatenate([X_train, X_min_noise])
y_train_augm = np.concatenate([y_train, y_min_noise])

In [13]:
# X_train_bal_augm = np.concatenate([X_balanced, X_min_noise])
# y_train_bal_augm = np.concatenate([y_balanced, y_min_noise])

In [20]:
#minirocket transformation with gaussian noise samples
miniRock.fit(X_train_augm)
X_train_new = miniRock.transform(X_train_augm)
X_test_new = miniRock.transform(X_test)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [9]:
# miniRock.fit(X_balanced)
# X_train_new = miniRock.transform(X_balanced)
# X_test_new = miniRock.transform(X_test)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [22]:
print(miniRock.num_kernels)
print(miniRock)

300
MiniRocket(n_jobs=-1, num_kernels=300, random_state=42)


In [10]:
# miniRock.fit(X_train_aug)
# X_train_new = miniRock.transform(X_train_aug)
# X_test_new = miniRock.transform(X_test)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [10]:
# miniRock.fit(X_train)
# X_train_new = miniRock.transform(X_train)
# X_test_new = miniRock.transform(X_test)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [16]:
np.save("../data/savedData/X_train_minirocket.npy", X_train_new)
np.save("../data/savedData/X_test_minirocket.npy",  X_test_new)

In [29]:
X_train_new = np.load("../data/savedData/X_train_minirocket.npy")
X_test_new  = np.load("../data/savedData/X_test_minirocket.npy")

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
# svc = LinearSVC(
#         max_iter=500,
#         C= 0.1,
#         class_weight= classWeight,
#         random_state=42,
#         dual=False

# )

In [30]:
#scaling data so features with large ranges dont dominate classification
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_new = scaler.fit_transform(X_train_new)
X_test_new = scaler.transform(X_test_new)

In [31]:
#chosen linear classifier which works on minirocket transformed features
from sklearn.linear_model import SGDClassifier

clf = SGDClassifier(
    loss="log_loss",         
    class_weight=classWeight,
    # class_weight = {0:1, 1:2},
    # class_weight= "balanced",
    max_iter= 3000,
    tol=1e-3,
    # alpha = 1e-4,
    # penalty = "l2",
    random_state=42
)

# clf = SGDClassifier(
#     loss="hinge",          # linear SVM
#     class_weight='balanced',
#     max_iter=2000,
#     tol=1e-3,
#     random_state=42
# )





In [22]:
# clf.fit(X_train_new, y_train)
# clf_pred = clf.predict(X_test_new)

In [12]:
# clf.fit(X_train_new, y_balanced)
# clf_pred = clf.predict(X_test_new)

In [32]:
clf.fit(X_train_new, y_train_augm)
clf_pred = clf.predict(X_test_new)

In [33]:
from sklearn.model_selection import FixedThresholdClassifier

clf.fit(X_train_new, y_train_augm)
thresh = FixedThresholdClassifier(clf, threshold=0.9)
clf_pred = thresh.predict(X_test_new)

In [34]:
#different thresholds tested
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

probs = clf.predict_proba(X_test_new)[:,1]

print("t | acc | prec_macro | rec_macro | f1_macro")

for t in [0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    
    preds = (probs > t).astype(int)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="macro")
    rec = recall_score(y_test, preds, average="macro")
    f1 = f1_score(y_test, preds, average="macro")

    print(f"{t:.1f} | {acc:.4f} | {prec:.4f} | {rec:.4f} | {f1:.4f}")

t | acc | prec_macro | rec_macro | f1_macro
0.3 | 0.9516 | 0.6167 | 0.8825 | 0.6707
0.4 | 0.9515 | 0.6162 | 0.8806 | 0.6699
0.5 | 0.9515 | 0.6162 | 0.8806 | 0.6699
0.6 | 0.9516 | 0.6163 | 0.8806 | 0.6701
0.7 | 0.9516 | 0.6165 | 0.8806 | 0.6703
0.8 | 0.9517 | 0.6166 | 0.8807 | 0.6705
0.9 | 0.9520 | 0.6171 | 0.8808 | 0.6712


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

probs = clf.predict_proba(X_test_new)[:,1]

print("t | acc | prec_macro | rec_macro | f1_macro")

for t in [0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    
    preds = (probs > t).astype(int)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="macro")
    rec = recall_score(y_test, preds, average="macro")
    f1 = f1_score(y_test, preds, average="macro")

    print(f"{t:.1f} | {acc:.4f} | {prec:.4f} | {rec:.4f} | {f1:.4f}")

t | acc | prec_macro | rec_macro | f1_macro
0.3 | 0.9569 | 0.6247 | 0.8640 | 0.6795
0.4 | 0.9571 | 0.6252 | 0.8641 | 0.6801
0.5 | 0.9571 | 0.6252 | 0.8641 | 0.6801
0.6 | 0.9572 | 0.6253 | 0.8641 | 0.6803
0.7 | 0.9573 | 0.6255 | 0.8642 | 0.6805
0.8 | 0.9573 | 0.6255 | 0.8642 | 0.6805
0.9 | 0.9574 | 0.6258 | 0.8642 | 0.6809


In [19]:
# probs = clf.predict_proba(X_test_new)[:,1]

# for t in [0.3,0.4,0.5,0.6, 0.7, 0.8,0.9]:
#     clf_pred = (probs > t).astype(int)
#     print(t, f1_score(y_test, clf_pred, average="macro"))

0.3 0.6802081847434692
0.4 0.68160424057421
0.5 0.6816117924766656
0.6 0.6830397111773437
0.7 0.6852161007954477
0.8 0.6862235846219135
0.9 0.6860500054264884


In [20]:
# clf.fit(X_train_new, y_train_bal_augm)
# clf_pred = clf.predict(X_test_new)

In [12]:
# clf.fit(X_train_new, y_train_aug)
# clf_pred = clf.predict(X_test_new)

In [ ]:
# clf.fit(X_train_bal, y_balanced)

,loss,'hinge'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,2000
,tol,0.001
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,None


In [38]:
#final results
from sklearn.metrics import classification_report
print(classification_report(y_test, clf_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9965    0.9545    0.9750     14448
           1     0.2378    0.8071    0.3674       254

    accuracy                         0.9520     14702
   macro avg     0.6171    0.8808    0.6712     14702
weighted avg     0.9834    0.9520    0.9645     14702



In [19]:
# svc = LinearSVC(
#         max_iter=500,
#         C= 0.1,
#         class_weight= classWeight,
#         random_state=42,
#         dual=False

# )

In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# calculates the values of the four evaluation metrics for the random forest 
accuracy = accuracy_score(y_test, clf_pred)

precision = precision_score(y_test, clf_pred, average = "macro")
recall = recall_score(y_test, clf_pred, average = "macro")
f1 = f1_score(y_test, clf_pred, average = "macro")

precision_w = precision_score(y_test, clf_pred, average = "weighted")
recall_w = recall_score(y_test, clf_pred, average = "weighted")
f1_w = f1_score(y_test, clf_pred, average = "weighted")

# displays results

print("Accuracy:", accuracy)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


print("Precision_w:", precision_w)
print("Recall_w:", recall_w)
print("F1-score_w:", f1_w)


Accuracy: 0.9519793225411509
Precision: 0.617139281547148
Recall: 0.880806596123159
F1-score: 0.6712129666796413
Precision_w: 0.9833528384795626
Recall_w: 0.9519793225411509
F1-score_w: 0.9645441646112081


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# calculates the values of the four evaluation metrics for the random forest 
accuracy = accuracy_score(y_test, clf_pred)

precision = precision_score(y_test, clf_pred, average = "macro")
recall = recall_score(y_test, clf_pred, average = "macro")
f1 = f1_score(y_test, clf_pred, average = "macro")

precision_w = precision_score(y_test, clf_pred, average = "weighted")
recall_w = recall_score(y_test, clf_pred, average = "weighted")
f1_w = f1_score(y_test, clf_pred, average = "weighted")

# displays results

print("Accuracy:", accuracy)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


print("Precision_w:", precision_w)
print("Recall_w:", recall_w)
print("F1-score_w:", f1_w)



Accuracy: 0.9574207590803973
Precision: 0.6258365435669178
Recall: 0.8642361746932796
F1-score: 0.6809033483150804
Precision_w: 0.9829853032420071
Recall_w: 0.9574207590803973
F1-score_w: 0.9676845939970113


In [ ]:
# Accuracy: 0.7943577648766328
# Precision: 0.4381251550075896
# Recall: 0.6431570072165511
# F1-score: 0.468059272904113
# Precision_w: 0.8455944266081156
# Recall_w: 0.7943577648766328
# F1-score_w: 0.8134937226499701

In [26]:
# Accuracy: 0.963270303360087
# Precision: 0.6315883695039652
# Recall: 0.8130632471813117
# F1-score: 0.6816117924766656
# Precision_w: 0.981305600907456
# Recall_w: 0.963270303360087
# F1-score_w: 0.9707252509290292

# alpha 1e-4

In [25]:
# Accuracy: 0.9519793225411509
# Precision: 0.617139281547148
# Recall: 0.880806596123159
# F1-score: 0.6712129666796413
# Precision_w: 0.9833528384795626
# Recall_w: 0.9519793225411509
# F1-score_w: 0.9645441646112081
# threshold 0.9

In [35]:
# Accuracy: 0.9443613113862059
# Precision: 0.6064041140208466
# Recall: 0.8943357007699619
# F1-score: 0.6572244193707669
# Precision_w: 0.9835832663564269
# Recall_w: 0.9443613113862059
# F1-score_w: 0.9601094554340206

In [ ]:
# Accuracy: 0.962794177662903
# Precision: 0.6355695193399259
# Recall: 0.8398955581133754
# F1-score: 0.6895217368727573
# Precision_w: 0.9823905169680873
# Recall_w: 0.962794177662903
# F1-score_w: 0.9707389502835692

# 0.02 noise level threshold 0.8

In [ ]:
# Accuracy: 0.9574207590803973
# Precision: 0.6258365435669178
# Recall: 0.8642361746932796
# F1-score: 0.6809033483150804
# Precision_w: 0.9829853032420071
# Recall_w: 0.9574207590803973
# F1-score_w: 0.9676845939970113

# 0.017 noise and 0.9 threshold

In [20]:
import pickle 

# saves minirocket 
with open('../models/minirocket.pkl', 'wb') as f:
    pickle.dump(miniRock, f)

# saves scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# saves sgdc classifier
with open('../models/sgdc.pkl', 'wb') as f:
    pickle.dump(clf, f)

print("Models Saved")


Models Saved


In [40]:
np.save("../data/savedData/y_test.npy", y_test)
np.save("../data/savedData/y_pred.npy", clf_pred)
np.save("../data/savedData/y_prob.npy", probs)
